# Quantitative Faithfulness



## 1. Configuration

In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FAITH_DIR = RESULTS_DIR / "faithfulness"
FAITH_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CSV = DATA_PROCESSED / "val.csv"

MASK_DIR = DATA_RAW / "HAM10000_segmentations_lesion_tschandl" / "HAM10000_segmentations_lesion_tschandl"

N_SAMPLES = 300

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_CONFIGS = {
    "cnn_baseline":  {"timm_name": "resnet50",             "description": "Baseline CNN (ResNet-50)"},
    "attention_cnn": {"timm_name": "resnet50",             "description": "ResNet-50 + CBAM"},
    "vit":           {"timm_name": "deit_small_patch16_224","description": "DeiT-Small (transformer)"},
}
DISPLAY_NAMES = {"cnn_baseline": "CNN", "attention_cnn": "Attention-CNN", "vit": "DeiT"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Mask folder exists:", MASK_DIR.exists())
if MASK_DIR.exists():
    sample_masks = list(MASK_DIR.glob("*.png"))[:3]
    print("Example mask files:", [p.name for p in sample_masks])
else:
    print("\nMask folder not found.")
    for p in sorted(DATA_RAW.iterdir()):
        print("  ", p.name)


## 2. Locate masks

In [ ]:
import pandas as pd

df = pd.read_csv(EVAL_CSV)

def find_mask(image_id):
    for pattern in [f"{image_id}_segmentation.png", f"{image_id}.png"]:
        p = MASK_DIR / pattern
        if p.exists():
            return p
    return None

df["mask_path"] = df["image_id"].apply(find_mask)
have_mask = df["mask_path"].notna()
print(f"Validation images: {len(df)}")
print(f"With a matching mask: {have_mask.sum()} ({100*have_mask.mean():.1f}%)")

if have_mask.sum() == 0:
    print("\nNo masks matched.")
    for p in list(MASK_DIR.glob('*'))[:5]:
        print("  ", p.name)
else:
    eval_df = df[have_mask].sample(min(N_SAMPLES, int(have_mask.sum())),
                                   random_state=42).reset_index(drop=True)
    print(f"\nEvaluating on {len(eval_df)} images "
          f"({eval_df['label'].sum()} melanoma, {(eval_df['label']==0).sum()} benign)")


## 3. Model definitions & loading

In [ ]:
import timm
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False),
                                 nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class AttentionCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.cbam = CBAM(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feats = self.cbam(self.backbone(x))
        return self.fc(self.dropout(self.pool(feats).flatten(1)))

class DropoutCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False,
                 dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(self.pool(self.backbone(x)).flatten(1)))


USE_DROPOUT_CNN = True

def build_model(model_key, num_classes=2):
    timm_name = MODEL_CONFIGS[model_key]["timm_name"]
    if model_key == "attention_cnn":
        return AttentionCNN(timm_name, num_classes, pretrained=False)
    if model_key == "cnn_baseline" and USE_DROPOUT_CNN:
        return DropoutCNN(timm_name, num_classes, pretrained=False)
    return timm.create_model(timm_name, pretrained=False, num_classes=num_classes)

def load_trained(model_key):
    ckpt = torch.load(MODELS_DIR / f"{model_key}_best.pth", map_location=device,
                      weights_only=False)
    model = build_model(model_key)
    model.load_state_dict(ckpt["model_state"] if "model_state" in ckpt else ckpt)
    model.to(device).eval()
    return model

print("Model definitions ready.")
print(f"USE_DROPOUT_CNN = {USE_DROPOUT_CNN} ")


## 4. Grad-CAM setup

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def deit_reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.permute(0, 3, 1, 2)

def get_cam(model_key, model):
    if model_key == "cnn_baseline":
        layers = [model.backbone.layer4[-1]] if USE_DROPOUT_CNN else [model.layer4[-1]]
        reshape = None
    elif model_key == "attention_cnn":
        layers = [model.backbone.layer4[-1]]
        reshape = None
    elif model_key == "vit":
        layers = [model.blocks[-1].norm1]
        reshape = deit_reshape_transform
    else:
        raise ValueError(model_key)
    return GradCAM(model=model, target_layers=layers, reshape_transform=reshape)

print("Grad-CAM setup ready.")


## 5. Faithfulness metrics


In [ ]:
import numpy as np
from PIL import Image
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image_and_mask(image_path, mask_path):
    tensor = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0)
    mask_img = Image.open(mask_path).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE),
                                                         Image.NEAREST)
    mask = np.array(mask_img) > 127         
    return tensor, mask

def inside_fraction(cam, mask):
    total = cam.sum()
    return float(cam[mask].sum() / total) if total > 0 else np.nan

def density_ratio(cam, mask):
    if mask.sum() == 0 or (~mask).sum() == 0:
        return np.nan
    outside = cam[~mask].mean()
    return float(cam[mask].mean() / outside) if outside > 0 else np.nan

print("Metric functions ready.")


## 6. Compute faithfulness for all three models

In [ ]:
records = []
print(f"Computing faithfulness over {len(eval_df)} images per model...")

for key in MODEL_CONFIGS:
    model = load_trained(key)
    cam_engine = get_cam(key, model)
    print(f"  {DISPLAY_NAMES[key]}...", end="", flush=True)

    for _, row in eval_df.iterrows():
        tensor, mask = load_image_and_mask(row["image_path"], row["mask_path"])
        if mask.sum() == 0:
            continue
        with torch.no_grad():
            prob = torch.softmax(model(tensor.to(device)).float(), dim=1)[0, 1].item()
        cam = cam_engine(input_tensor=tensor.to(device),
                         targets=[ClassifierOutputTarget(1)])[0]
        records.append({
            "model": DISPLAY_NAMES[key],
            "image_id": row["image_id"],
            "label": int(row["label"]),
            "pred": int(prob >= 0.5),
            "prob": prob,
            "inside_fraction": inside_fraction(cam, mask),
            "density_ratio": density_ratio(cam, mask),
            "lesion_area_frac": float(mask.mean()),
        })

    print(" done")
    del model, cam_engine
    if device.type == "cuda":
        torch.cuda.empty_cache()

faith_df = pd.DataFrame(records)
faith_df.to_csv(FAITH_DIR / "faithfulness_per_image.csv", index=False)
print(f"\nSaved per-image results: {len(faith_df)} rows")


## 7. Headline faithfulness table


In [ ]:
summary = faith_df.groupby("model").agg(
    n=("image_id", "count"),
    inside_fraction_mean=("inside_fraction", "mean"),
    inside_fraction_std=("inside_fraction", "std"),
    density_ratio_mean=("density_ratio", "mean"),
    density_ratio_median=("density_ratio", "median"),
).round(3)

order = [DISPLAY_NAMES[k] for k in MODEL_CONFIGS]
summary = summary.reindex([m for m in order if m in summary.index])
summary.to_csv(FAITH_DIR / "faithfulness_summary.csv")
print(summary.to_string())
print("\nMean lesion area fraction: "
      f"{faith_df['lesion_area_frac'].mean():.3f} "
)
summary


## 8. Does faithfulness differ between correct and incorrect predictions?

In [ ]:
faith_df["outcome"] = np.where(
    (faith_df.label == 0) & (faith_df.pred == 1), "false positive",
    np.where((faith_df.label == 1) & (faith_df.pred == 0), "false negative", "correct"))

outcome_table = (faith_df.groupby(["model", "outcome"])["density_ratio"]
                 .agg(["count", "mean"]).round(3))
outcome_table.to_csv(FAITH_DIR / "faithfulness_by_outcome.csv")
print(outcome_table.to_string())


## 9. Figures

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data = [faith_df[faith_df.model == m]["density_ratio"].dropna()
        for m in summary.index]
axes[0].boxplot(data, tick_labels=list(summary.index), showfliers=False)
axes[0].axhline(1.0, color="red", linestyle="--", alpha=0.6,
                label="1.0 = attention spread evenly")
axes[0].set_ylabel("Density ratio (inside ÷ outside lesion)")
axes[0].set_title("Lesion-focus of explanations by architecture")
axes[0].legend(fontsize=9)

outcomes = ["correct", "false positive", "false negative"]
width = 0.25
x = np.arange(len(summary.index))
for i, oc in enumerate(outcomes):
    means = [faith_df[(faith_df.model == m) & (faith_df.outcome == oc)]["density_ratio"].mean()
             for m in summary.index]
    axes[1].bar(x + i * width, means, width, label=oc)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(list(summary.index))
axes[1].axhline(1.0, color="red", linestyle="--", alpha=0.6)
axes[1].set_ylabel("Mean density ratio")
axes[1].set_title("Lesion-focus by prediction outcome")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FAITH_DIR / "faithfulness_comparison.png", dpi=150)
plt.show()
print(f"Saved to {FAITH_DIR}")
